### PathVQA Data Filtering for Histopathology Images

This code will filter down all the histopathology images and relevant questions from the PathVQA dataset since PathVQA contains gross and histopathology images. But we only want to do the evaluation on histopathology images

In [1]:
import sys
from PIL import Image
import pandas as pd
import os
import pickle
from collections import Counter
import numpy as np
import shutil

sys.path.append(os.path.join(os.getcwd(), 'histocartography'))
from histocartography.preprocessing import NucleiExtractor

/data/mn27889/miniconda3/envs/path-opendata/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Defining a function to detect the nuclei in the image

In [2]:
# Cell Graph Generation Definitions
nuclei_detector = NucleiExtractor()

def detect_histopathology_images(img_path):    
    query_img = Image.open(img_path).convert(mode="RGB")
    image = np.array(query_img)
    nuclei_map, nuclei_centers = nuclei_detector.process(image)

    # Only consider if more than 5 nuclei are detected since knn needs to form a graph using 5 neighbors.
    # If less than 5 nuclei are present, most of the images are not pathology related
    if nuclei_centers.shape[0] > 10:
        return True
    else:
        return False

File already downloaded.


/data/mn27889/path-open-data/histocartography/histocartography/preprocessing/nuclei_extraction.py:88: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model = torch.load(m

#### Defining the paths for PVQA dataset

In [3]:
pvqa_data_path = "/data/mn27889/pvqa"
pvqa_images = os.path.join(pvqa_data_path, "images")
pvqa_qas = os.path.join(pvqa_data_path, "qas")

### Defining the paths of PathVQA images to filter down the histopathology images from gross images

In [4]:
pvqa_histo_data_path = "/data/mn27889/path-open-data/pathvqa-histopathology"
pvqa_histo_images = os.path.join(pvqa_histo_data_path, "images")
pvqa_histo_qas = os.path.join(pvqa_histo_data_path, "qas")
os.makedirs(pvqa_histo_images, exist_ok=True)
os.makedirs(pvqa_histo_qas, exist_ok=True)

#### 1. Filtering the images/qas from `train` subset

Defining the source directories

In [5]:
pvqa_subset = "train"
pvqa_images_subset_path = os.path.join(pvqa_images, pvqa_subset)
pvqa_qas_subset_path = os.path.join(pvqa_qas, pvqa_subset)

Defining the destination directories

In [6]:
pvqa_histo_images_subset_path = os.path.join(pvqa_histo_images, pvqa_subset)
pvqa_histo_qas_subset_path = os.path.join(pvqa_histo_qas, pvqa_subset)
os.makedirs(pvqa_histo_images_subset_path, exist_ok=True)
os.makedirs(pvqa_histo_qas_subset_path, exist_ok=True)

Detecting all the histopathology images

In [7]:
image_list = os.listdir(pvqa_images_subset_path)
histo_images = []
for image in image_list:
    image_path = os.path.join(pvqa_images_subset_path, image)
    is_image_histo = detect_histopathology_images(image_path)
    print(image)
    if is_image_histo:
        histo_images.append(image)

Patch-level nuclei detection:   0%|          | 0/2 [00:00<?, ?it/s]/data/mn27889/path-open-data/histocartography/histocartography/ml/models/hovernet.py:233: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  x = F.upsample(x, scale_factor=2, mode='nearest')
Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  2.29it/s]


train_1436.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  4.98it/s]


train_0345.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.12it/s]


...

train_0769.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.22it/s]


train_1452.jpg


Patch-level nuclei detection: 100%|██████████| 1/1 [00:00<00:00,  4.37it/s]

train_0023.jpg


Filtering all the questions related to histopathology images

In [8]:
file_name = "train_qa.pkl"
qas_file_path = os.path.join(pvqa_qas_subset_path, file_name)
with open(qas_file_path, 'rb') as file:
    pvqa_qas_subset = pickle.load(file)

histo_images_without_extension = [os.path.splitext(img_name)[0] for img_name in histo_images]
pvqa_histo_qas_subset = [qa_sample for qa_sample in pvqa_qas_subset if qa_sample['image'] in histo_images_without_extension]

Moving all the histo images to destination directory

In [9]:
for image_name in histo_images:
    src_image_path = os.path.join(pvqa_images_subset_path, image_name)
    shutil.copy(src_image_path, pvqa_histo_images_subset_path)

Moving the updated qa_samples to destination directory

In [10]:
pvqa_histo_qas_subset_path_file = os.path.join(pvqa_histo_qas_subset_path, file_name)
with open(pvqa_histo_qas_subset_path_file, 'wb') as file:
        pickle.dump(pvqa_histo_qas_subset, file)

#### 2. Filtering the images/qas from `test` subset

Defining the source directories

In [11]:
pvqa_subset = "test"
pvqa_images_subset_path = os.path.join(pvqa_images, pvqa_subset)
pvqa_qas_subset_path = os.path.join(pvqa_qas, pvqa_subset)

Defining the destination directories

In [12]:
pvqa_histo_images_subset_path = os.path.join(pvqa_histo_images, pvqa_subset)
pvqa_histo_qas_subset_path = os.path.join(pvqa_histo_qas, pvqa_subset)
os.makedirs(pvqa_histo_images_subset_path, exist_ok=True)
os.makedirs(pvqa_histo_qas_subset_path, exist_ok=True)

Detecting all the histopathology images

In [13]:
image_list = os.listdir(pvqa_images_subset_path)
histo_images = []
for image in image_list:
    image_path = os.path.join(pvqa_images_subset_path, image)
    is_image_histo = detect_histopathology_images(image_path)
    print(image)
    if is_image_histo:
        histo_images.append(image)

Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  4.68it/s]


test_0478.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.25it/s]


test_0837.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.25it/s]


...

test_0784.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.21it/s]


test_0672.jpg


Patch-level nuclei detection: 100%|██████████| 4/4 [00:01<00:00,  3.71it/s]


test_0261.jpg


Filtering all the questions related to histopathology images

In [14]:
file_name = "test_qa.pkl"
qas_file_path = os.path.join(pvqa_qas_subset_path, file_name)
with open(qas_file_path, 'rb') as file:
    pvqa_qas_subset = pickle.load(file)

histo_images_without_extension = [os.path.splitext(img_name)[0] for img_name in histo_images]
pvqa_histo_qas_subset = [qa_sample for qa_sample in pvqa_qas_subset if qa_sample['image'] in histo_images_without_extension]

Moving all the histo images to destination directory

In [15]:
for image_name in histo_images:
    src_image_path = os.path.join(pvqa_images_subset_path, image_name)
    shutil.copy(src_image_path, pvqa_histo_images_subset_path)

Moving the updated qa_samples to destination directory

In [16]:
pvqa_histo_qas_subset_path_file = os.path.join(pvqa_histo_qas_subset_path, file_name)
with open(pvqa_histo_qas_subset_path_file, 'wb') as file:
        pickle.dump(pvqa_histo_qas_subset, file)

#### 3. Filtering the images/qas from `val` subset

Defining the source directories

In [17]:
pvqa_subset = "val"
pvqa_images_subset_path = os.path.join(pvqa_images, pvqa_subset)
pvqa_qas_subset_path = os.path.join(pvqa_qas, pvqa_subset)

Defining the destination directories

In [18]:
pvqa_histo_images_subset_path = os.path.join(pvqa_histo_images, pvqa_subset)
pvqa_histo_qas_subset_path = os.path.join(pvqa_histo_qas, pvqa_subset)
os.makedirs(pvqa_histo_images_subset_path, exist_ok=True)
os.makedirs(pvqa_histo_qas_subset_path, exist_ok=True)

Detecting all the histopathology images

In [19]:
image_list = os.listdir(pvqa_images_subset_path)
histo_images = []
for image in image_list:
    image_path = os.path.join(pvqa_images_subset_path, image)
    is_image_histo = detect_histopathology_images(image_path)
    print(image)
    if is_image_histo:
        histo_images.append(image)

Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  4.56it/s]


val_0872.jpg


Patch-level nuclei detection: 100%|██████████| 1/1 [00:00<00:00,  4.38it/s]


val_0330.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.23it/s]


...

val_0350.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.22it/s]


val_0393.jpg


Patch-level nuclei detection: 100%|██████████| 2/2 [00:00<00:00,  5.12it/s]

val_0571.jpg


Filtering all the questions related to histopathology images

In [20]:
file_name = "val_qa.pkl"
qas_file_path = os.path.join(pvqa_qas_subset_path, file_name)
with open(qas_file_path, 'rb') as file:
    pvqa_qas_subset = pickle.load(file)

histo_images_without_extension = [os.path.splitext(img_name)[0] for img_name in histo_images]
pvqa_histo_qas_subset = [qa_sample for qa_sample in pvqa_qas_subset if qa_sample['image'] in histo_images_without_extension]

Moving all the histo images to destination directory

In [21]:
for image_name in histo_images:
    src_image_path = os.path.join(pvqa_images_subset_path, image_name)
    shutil.copy(src_image_path, pvqa_histo_images_subset_path)

Moving the updated qa_samples to destination directory

In [22]:
pvqa_histo_qas_subset_path_file = os.path.join(pvqa_histo_qas_subset_path, file_name)
with open(pvqa_histo_qas_subset_path_file, 'wb') as file:
        pickle.dump(pvqa_histo_qas_subset, file)